# GPSR Training Data Generator

Generates `gpsr_train.jsonl` in the same `{"messages": [...]}` format as `oldGPSR_testset.jsonl`, covering all 25 action classes found in that file's schema.

**No GPU needed** — pure Python, runs in seconds.

**Anti-leakage design:** open-vocabulary entity pools (locations, names, objects, clothing) are split into a `TRAIN` pool and a `HOLDOUT` pool up front. Only `TRAIN` values are used to generate `gpsr_train.jsonl`. This guarantees the model can't have memorized any specific location/name/object pairing that also happens to appear in the team's real eval set — it has to actually parse the sentence structure. Closed, small vocabularies (poses, genders, a handful of gestures/talk topics) are shared, since there's nothing meaningful to leak there.

**Known bug fixed:** the old test set had `source_location == target_location` on every `takeAndPlaceObj` example (the true source was never captured). This generator always samples two *different* locations for source/target.

In [ ]:
import json
import random

random.seed(42)  # reproducible generation

## 1. Entity pools (train vs holdout split)

In [ ]:
# --- open-vocab pools, split train / holdout so nothing overlaps with the team's eval set ---
_locations_all = [
    "bathroom sink", "hallway shelf", "living room couch", "bookshelf", "side table",
    "nightstand", "tv stand", "kitchen table", "bedroom desk", "dining table",
    "garden bench", "counter", "cabinet", "entrance door", "office chair",
    "kitchen counter", "living room shelf", "study desk", "bedroom nightstand",
    "dining room cabinet", "garage shelf", "hallway table", "office desk",
]
random.shuffle(_locations_all)
_split = int(len(_locations_all) * 0.8)
LOCATIONS_TRAIN, LOCATIONS_HOLDOUT = _locations_all[:_split], _locations_all[_split:]

_rooms_all = [
    "bathroom", "hallway", "living room", "study", "office", "kitchen",
    "bedroom", "dining room", "garden", "garage",
]
random.shuffle(_rooms_all)
_split = int(len(_rooms_all) * 0.8)
ROOMS_TRAIN, ROOMS_HOLDOUT = _rooms_all[:_split], _rooms_all[_split:]

_names_all = [
    "Nancy", "Fiona", "Julia", "Ivan", "Charlie", "Hannah", "Edward",
    "Kevin", "George", "Bob", "Diana", "Oscar", "Laura", "Peter", "Susan",
]
random.shuffle(_names_all)
_split = int(len(_names_all) * 0.8)
NAMES_TRAIN, NAMES_HOLDOUT = _names_all[:_split], _names_all[_split:]

_objects_all = [
    "wallet", "spoon", "bowl", "banana", "coke", "knife", "tablet",
    "butter", "jam", "water", "apple", "fork", "laptop", "cracker",
    "juice", "plate", "orange", "phone", "cereal", "milk",
]
random.shuffle(_objects_all)
_split = int(len(_objects_all) * 0.8)
OBJECTS_TRAIN, OBJECTS_HOLDOUT = _objects_all[:_split], _objects_all[_split:]

_clothing_all = [
    "yellow sweater", "gray t shirt", "black shirt", "orange blouse",
    "white jacket", "gray sweater", "red coat", "yellow t shirt",
    "white shirt", "black coat", "orange sweater", "gray blouse",
    "blue jacket", "green shirt",
]
random.shuffle(_clothing_all)
_split = int(len(_clothing_all) * 0.8)
CLOTHING_TRAIN, CLOTHING_HOLDOUT = _clothing_all[:_split], _clothing_all[_split:]

# --- closed, small vocabularies: shared (nothing meaningful to leak) ---
OBJECT_CATEGORIES = ["utensil", "snack", "drink", "electronic", "stationery"]
OBJECT_COMPARATORS = ["biggest", "smallest", "heaviest", "lightest", "thinnest", "largest"]
POSES = ["sitting person", "standing person", "lying person"]
GESTURES = [
    "person raising their left arm", "person raising their right arm",
    "person pointing to the left", "person pointing to the right", "waving person",
]
PERSON_INFO = ["pose", "name"]
TALK_TOPICS = ["the time", "the day of the week", "something about yourself", "the date"]

print(f"locations: {len(LOCATIONS_TRAIN)} train / {len(LOCATIONS_HOLDOUT)} holdout")
print(f"rooms:     {len(ROOMS_TRAIN)} train / {len(ROOMS_HOLDOUT)} holdout")
print(f"names:     {len(NAMES_TRAIN)} train / {len(NAMES_HOLDOUT)} holdout")
print(f"objects:   {len(OBJECTS_TRAIN)} train / {len(OBJECTS_HOLDOUT)} holdout")
print(f"clothing:  {len(CLOTHING_TRAIN)} train / {len(CLOTHING_HOLDOUT)} holdout")

## 2. One generator function per action class
Each returns `(sentence_fragment, slot_dict)`. `slot_dict` always includes `"action"`; `"sequence"` is added later when steps are assembled into a full command.

In [ ]:
def _pick(pool):
    return random.choice(pool)

def _two_distinct(pool):
    a, b = random.sample(pool, 2) if len(pool) >= 2 else (pool[0], pool[0])
    return a, b

def gen_tellPrsInfoAtLocToPrsAtLoc(loc, room, name, obj, cat, comp, cloth, pose, gest, info, topic):
    frm, to = _two_distinct(loc)
    info_choice = _pick(PERSON_INFO)
    tmpl = random.choice([
        f"tell the person at the {to} the {info_choice} of the person near the {frm}",
        f"let the person at the {to} know the {info_choice} of the individual by the {frm}",
    ])
    return tmpl, {"action": "tellPrsInfoAtLocToPrsAtLoc", "from_location": frm, "to_location": to, "person_info": info_choice}

def gen_findAndFollowPrsInRoom(loc, room, name, obj, cat, comp, cloth, pose, gest, info, topic):
    r = _pick(room)
    use_gesture = random.random() < 0.5
    descriptor = _pick(GESTURES) if use_gesture else _pick(POSES)
    tmpl = random.choice([
        f"find and follow the {descriptor} in the {r}",
        f"keep an eye on the {descriptor} in the {r}",
    ])
    slots = {"action": "findAndFollowPrsInRoom", "to_location": r, "room": r}
    slots["gesture" if use_gesture else "pose"] = descriptor
    return tmpl, slots

def gen_meetNameAtLocThenFindInRm(loc, room, name, obj, cat, comp, cloth, pose, gest, info, topic):
    l, r, n = _pick(loc), _pick(room), _pick(name)
    tmpl = f"meet {n} at the {l} then find them in the {r}"
    return tmpl, {"action": "meetNameAtLocThenFindInRm", "meet_location": l, "find_room": r, "person_name": n}

def gen_takeAndPlaceObj(loc, room, name, obj, cat, comp, cloth, pose, gest, info, topic):
    src, tgt = _two_distinct(loc)   # FIX: always distinct, unlike the old buggy label
    o = _pick(obj)
    tmpl = f"go to the {src}, grab the {o}, and put it on the {tgt}"
    return tmpl, {"action": "takeAndPlaceObj", "source_location": src, "target_location": tgt, "object": o}

def gen_tellPrsInfoInLoc(loc, room, name, obj, cat, comp, cloth, pose, gest, info, topic):
    use_room = random.random() < 0.5
    place = _pick(room) if use_room else _pick(loc)
    info_choice = _pick(PERSON_INFO)
    tmpl = f"tell me the {info_choice} of the person in the {place}"
    slots = {"action": "tellPrsInfoInLoc", "person_info": info_choice}
    slots["room" if use_room else "location"] = place
    return tmpl, slots

def gen_followNameFromBeacToRoom(loc, room, name, obj, cat, comp, cloth, pose, gest, info, topic):
    n, r = _pick(name), _pick(room)
    include_from = random.random() < 0.6
    slots = {"action": "followNameFromBeacToRoom", "to_room": r, "person_name": n}
    if include_from:
        f = _pick(loc)
        slots["from_location"] = f
        tmpl = f"follow {n} from the {f} to the {r}"
    else:
        tmpl = f"follow {n} to the {r}"
    return tmpl, slots

def gen_tellObjPropOnPlcmt(loc, room, name, obj, cat, comp, cloth, pose, gest, info, topic):
    c, p = _pick(comp), _pick(loc)
    tmpl = f"what's the {c} thing on the {p}"
    return tmpl, {"action": "tellObjPropOnPlcmt", "object_comparator": c, "placement_location": p}

def gen_greetClothDscInRm(loc, room, name, obj, cat, comp, cloth, pose, gest, info, topic):
    r, c = _pick(room), _pick(cloth)
    tmpl = f"greet the person wearing a {c} in the {r}"
    return tmpl, {"action": "greetClothDscInRm", "room": r, "clothing": c}

def gen_navigateToLocation(loc, room, name, obj, cat, comp, cloth, pose, gest, info, topic):
    use_room = random.random() < 0.5
    place = _pick(room) if use_room else _pick(loc)
    tmpl = random.choice([f"go to the {place}", f"move to the {place}", f"head to the {place}"])
    slots = {"action": "navigateToLocation"}
    slots["room" if use_room else "location"] = place
    return tmpl, slots

def gen_countClothPrsInRoom(loc, room, name, obj, cat, comp, cloth, pose, gest, info, topic):
    r, c = _pick(room), _pick(cloth)
    tmpl = f"how many people in the {r} are wearing a {c}"
    return tmpl, {"action": "countClothPrsInRoom", "room": r, "clothing": c}

def gen_countPrsInRoom(loc, room, name, obj, cat, comp, cloth, pose, gest, info, topic):
    r = _pick(room)
    include_descriptor = random.random() < 0.6
    slots = {"action": "countPrsInRoom", "room": r}
    if include_descriptor:
        use_gesture = random.random() < 0.3
        d = _pick(GESTURES) if use_gesture else _pick(POSES)
        slots["gesture" if use_gesture else "pose"] = d
        tmpl = f"how many {d}s are in the {r}"
    else:
        tmpl = f"how many people are in the {r}"
    return tmpl, slots

def gen_talkInfoToGestPrsInRoom(loc, room, name, obj, cat, comp, cloth, pose, gest, info, topic):
    r, t = _pick(room), _pick(topic)
    use_gesture = random.random() < 0.7
    d = _pick(GESTURES) if use_gesture else _pick(POSES)
    tmpl = f"tell the {d} in the {r} {t}"
    slots = {"action": "talkInfoToGestPrsInRoom", "room": r, "talk_topic": t}
    slots["gesture" if use_gesture else "pose"] = d
    return tmpl, slots

def gen_deliverObjToPrs(loc, room, name, obj, cat, comp, cloth, pose, gest, info, topic):
    r = _pick(room)
    use_cat = random.random() < 0.5
    o = _pick(cat) if use_cat else _pick(obj)
    target_kind = random.choice(["name", "pose", "gesture"])
    slots = {"action": "deliverObjToPrs", "target_room": r}
    slots["object_category" if use_cat else "object"] = o
    if random.random() < 0.6:
        s = _pick(loc)
        slots["source_location"] = s
        src_phrase = f"from the {s} "
    else:
        src_phrase = ""
    if target_kind == "name":
        n = _pick(name)
        slots["target_name"] = n
        target_phrase = f"to {n} in the {r}"
    elif target_kind == "pose":
        p = _pick(POSES)
        slots["target_pose"] = p
        target_phrase = f"to the {p} in the {r}"
    else:
        g = _pick(GESTURES)
        slots["target_gesture"] = g
        target_phrase = f"to the {g} in the {r}"
    tmpl = f"bring the {o} {src_phrase}{target_phrase}"
    return tmpl, slots

def gen_guideNameFromBeacToBeac(loc, room, name, obj, cat, comp, cloth, pose, gest, info, topic):
    n, t = _pick(name), _pick(loc)
    slots = {"action": "guideNameFromBeacToBeac", "to_location": t, "person_name": n}
    if random.random() < 0.6:
        f = _pick(loc)
        slots["from_location"] = f
        tmpl = f"guide {n} from the {f} to the {t}"
    else:
        tmpl = f"guide {n} to the {t}"
    return tmpl, slots

def gen_countObjOnPlcmt(loc, room, name, obj, cat, comp, cloth, pose, gest, info, topic):
    c, p = _pick(cat), _pick(loc)
    tmpl = f"how many {c}s are on the {p}"
    return tmpl, {"action": "countObjOnPlcmt", "object_category": c, "placement_location": p}

def gen_guidePrsFromBeacToBeac(loc, room, name, obj, cat, comp, cloth, pose, gest, info, topic):
    f, t = _two_distinct(loc)
    use_gesture = random.random() < 0.5
    d = _pick(GESTURES) if use_gesture else _pick(POSES)
    tmpl = f"guide the {d} from the {f} to the {t}"
    slots = {"action": "guidePrsFromBeacToBeac", "from_location": f, "to_location": t}
    slots["gesture" if use_gesture else "pose"] = d
    return tmpl, slots

def gen_followPrsAtLoc(loc, room, name, obj, cat, comp, cloth, pose, gest, info, topic):
    slots = {"action": "followPrsAtLoc"}
    if random.random() < 0.4:
        l = _pick(loc)
        slots["location"] = l
        tmpl = f"follow the person near the {l}"
    else:
        tmpl = "follow the person"
    return tmpl, slots

def gen_answerToGestPrsInRoom(loc, room, name, obj, cat, comp, cloth, pose, gest, info, topic):
    r = _pick(room)
    use_gesture = random.random() < 0.7
    d = _pick(GESTURES) if use_gesture else _pick(POSES)
    slots = {"action": "answerToGestPrsInRoom"}
    if random.random() < 0.85:
        slots["room"] = r
        tmpl = f"answer the question from the {d} in the {r}"
    else:
        tmpl = f"answer the question from the {d}"
    slots["gesture" if use_gesture else "pose"] = d
    return tmpl, slots

def gen_greetNameInRm(loc, room, name, obj, cat, comp, cloth, pose, gest, info, topic):
    r, n = _pick(room), _pick(name)
    tmpl = f"greet {n} in the {r}"
    return tmpl, {"action": "greetNameInRm", "room": r, "person_name": n}

def gen_tellCatPropOnPlcmt(loc, room, name, obj, cat, comp, cloth, pose, gest, info, topic):
    c, cm, p = _pick(cat), _pick(comp), _pick(loc)
    tmpl = f"which {c} is the {cm} on the {p}"
    return tmpl, {"action": "tellCatPropOnPlcmt", "object_category": c, "object_comparator": cm, "placement_location": p}

def gen_findPrsInRoom(loc, room, name, obj, cat, comp, cloth, pose, gest, info, topic):
    r = _pick(room)
    use_gesture = random.random() < 0.5
    d = _pick(GESTURES) if use_gesture else _pick(POSES)
    tmpl = f"find the {d} in the {r}"
    slots = {"action": "findPrsInRoom", "room": r}
    slots["gesture" if use_gesture else "pose"] = d
    return tmpl, slots

def gen_guideClothPrsFromBeacToBeac(loc, room, name, obj, cat, comp, cloth, pose, gest, info, topic):
    t, c = _pick(loc), _pick(cloth)
    slots = {"action": "guideClothPrsFromBeacToBeac", "to_location": t, "clothing": c}
    if random.random() < 0.6:
        f = _pick(loc)
        while f == t:
            f = _pick(loc)
        slots["from_location"] = f
        tmpl = f"guide the person in the {c} from the {f} to the {t}"
    else:
        tmpl = f"guide the person in the {c} to the {t}"
    return tmpl, slots

def gen_meetPrsAtBeac(loc, room, name, obj, cat, comp, cloth, pose, gest, info, topic):
    n = _pick(name)
    slots = {"action": "meetPrsAtBeac", "person_name": n}
    if random.random() < 0.4:
        l = _pick(loc)
        slots["location"] = l
        tmpl = f"meet {n} at the {l}"
    else:
        tmpl = f"meet {n}"
    return tmpl, slots

def gen_bringMeObjFromPlcmt(loc, room, name, obj, cat, comp, cloth, pose, gest, info, topic):
    use_cat = random.random() < 0.3
    o = _pick(cat) if use_cat else _pick(obj)
    slots = {"action": "bringMeObjFromPlcmt"}
    slots["object_category" if use_cat else "object"] = o
    if random.random() < 0.7:
        s = _pick(loc)
        slots["source_location"] = s
        tmpl = f"bring me the {o} from the {s}"
    else:
        tmpl = f"bring me a {o}"
    return tmpl, slots

def gen_findAndGuidePrsToRoom(loc, room, name, obj, cat, comp, cloth, pose, gest, info, topic):
    r, tr = _two_distinct(room)
    use_gesture = random.random() < 0.5
    d = _pick(GESTURES) if use_gesture else _pick(POSES)
    tmpl = f"find the {d} in the {r} and guide them to the {tr}"
    slots = {"action": "findAndGuidePrsToRoom", "room": r, "to_room": tr}
    slots["gesture" if use_gesture else "pose"] = d
    return tmpl, slots

ACTION_GENERATORS = {
    "tellPrsInfoAtLocToPrsAtLoc": gen_tellPrsInfoAtLocToPrsAtLoc,
    "findAndFollowPrsInRoom": gen_findAndFollowPrsInRoom,
    "meetNameAtLocThenFindInRm": gen_meetNameAtLocThenFindInRm,
    "takeAndPlaceObj": gen_takeAndPlaceObj,
    "tellPrsInfoInLoc": gen_tellPrsInfoInLoc,
    "followNameFromBeacToRoom": gen_followNameFromBeacToRoom,
    "tellObjPropOnPlcmt": gen_tellObjPropOnPlcmt,
    "greetClothDscInRm": gen_greetClothDscInRm,
    "navigateToLocation": gen_navigateToLocation,
    "countClothPrsInRoom": gen_countClothPrsInRoom,
    "countPrsInRoom": gen_countPrsInRoom,
    "talkInfoToGestPrsInRoom": gen_talkInfoToGestPrsInRoom,
    "deliverObjToPrs": gen_deliverObjToPrs,
    "guideNameFromBeacToBeac": gen_guideNameFromBeacToBeac,
    "countObjOnPlcmt": gen_countObjOnPlcmt,
    "guidePrsFromBeacToBeac": gen_guidePrsFromBeacToBeac,
    "followPrsAtLoc": gen_followPrsAtLoc,
    "answerToGestPrsInRoom": gen_answerToGestPrsInRoom,
    "greetNameInRm": gen_greetNameInRm,
    "tellCatPropOnPlcmt": gen_tellCatPropOnPlcmt,
    "findPrsInRoom": gen_findPrsInRoom,
    "guideClothPrsFromBeacToBeac": gen_guideClothPrsFromBeacToBeac,
    "meetPrsAtBeac": gen_meetPrsAtBeac,
    "bringMeObjFromPlcmt": gen_bringMeObjFromPlcmt,
    "findAndGuidePrsToRoom": gen_findAndGuidePrsToRoom,
}
print(f"{len(ACTION_GENERATORS)} action generators defined")

## 3. Compose multi-step commands + build the JSONL dataset

In [ ]:
CONNECTORS = [", then ", " and then ", ", and ", ". Then "]

def make_command(pools, n_steps, action_names=None):
    """Build one command with n_steps actions, sampled without repetition within the command."""
    names = action_names or list(ACTION_GENERATORS.keys())
    chosen = random.sample(names, n_steps)
    fragments, steps = [], []
    for i, action_name in enumerate(chosen, start=1):
        frag, slots = ACTION_GENERATORS[action_name](
            pools["loc"], pools["room"], pools["name"], pools["obj"],
            OBJECT_CATEGORIES, OBJECT_COMPARATORS, pools["cloth"],
            POSES, GESTURES, PERSON_INFO, TALK_TOPICS,
        )
        slots = {"sequence": i, **slots}
        fragments.append(frag)
        steps.append(slots)
    sentence = fragments[0]
    for frag in fragments[1:]:
        sentence += random.choice(CONNECTORS) + frag
    sentence = sentence[0].upper() + sentence[1:]
    return sentence, steps

def build_dataset(pools, n_examples, steps_distribution=(1, 2, 3), weights=(0.34, 0.33, 0.33)):
    rows = []
    for _ in range(n_examples):
        n_steps = random.choices(steps_distribution, weights=weights, k=1)[0]
        sentence, steps = make_command(pools, n_steps)
        rows.append({
            "messages": [
                {"role": "user", "content": f"Parse this robot command:\n{sentence}"},
                {"role": "assistant", "content": json.dumps(steps, ensure_ascii=False)},
            ]
        })
    return rows

TRAIN_POOLS = {
    "loc": LOCATIONS_TRAIN, "room": ROOMS_TRAIN, "name": NAMES_TRAIN,
    "obj": OBJECTS_TRAIN, "cloth": CLOTHING_TRAIN,
}
HOLDOUT_POOLS = {
    "loc": LOCATIONS_HOLDOUT, "room": ROOMS_HOLDOUT, "name": NAMES_HOLDOUT,
    "obj": OBJECTS_HOLDOUT, "cloth": CLOTHING_HOLDOUT,
}

N_TRAIN = 2000   # bump this up if you want more data; generation is cheap
N_HOLDOUT = 150  # a self-generated sanity-check set, separate from oldGPSR_testset.jsonl

train_rows = build_dataset(TRAIN_POOLS, N_TRAIN)
holdout_rows = build_dataset(HOLDOUT_POOLS, N_HOLDOUT)

with open("gpsr_train.jsonl", "w", encoding="utf-8") as f:
    for row in train_rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

with open("gpsr_holdout.jsonl", "w", encoding="utf-8") as f:
    for row in holdout_rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print(f"Wrote {len(train_rows)} rows -> gpsr_train.jsonl")
print(f"Wrote {len(holdout_rows)} rows -> gpsr_holdout.jsonl (extra sanity-check set, distinct entities)")
print("\nSample training row:")
print(json.dumps(train_rows[0], indent=2, ensure_ascii=False))

## 4. Sanity checks before you train on this

In [ ]:
from collections import Counter

action_counts = Counter()
step_counts = Counter()
for row in train_rows:
    steps = json.loads(row["messages"][1]["content"])
    step_counts[len(steps)] += 1
    for s in steps:
        action_counts[s["action"]] += 1

print("Action class balance (train):")
for a, c in action_counts.most_common():
    print(f"  {a}: {c}")

print("\nStep-count distribution (train):")
for k in sorted(step_counts):
    print(f"  {k}-step: {step_counts[k]}")

# confirm zero entity overlap between train and holdout pools
overlap = (set(LOCATIONS_TRAIN) & set(LOCATIONS_HOLDOUT)) | (set(NAMES_TRAIN) & set(NAMES_HOLDOUT)) \
    | (set(OBJECTS_TRAIN) & set(OBJECTS_HOLDOUT)) | (set(CLOTHING_TRAIN) & set(CLOTHING_HOLDOUT)) \
    | (set(ROOMS_TRAIN) & set(ROOMS_HOLDOUT))
print(f"\nEntity overlap between train and holdout pools: {overlap if overlap else 'NONE (good)'}")

## 5. Next step
Download `gpsr_train.jsonl` (Colab file panel, right-click > Download) and upload it into the fine-tuning notebook's Section 4. Keep `gpsr_holdout.jsonl` around too — useful as an extra sanity check before you run the full eval against `oldGPSR_testset.jsonl`, since its entities never appeared during training.